# 04 · Selección y registro del modelo

**Propósito.** Verificar que el mejor pipeline de cada familia quedó registrado como una versión del **mismo** modelo de Unity Catalog, con firma, ejemplo de entrada y trazabilidad a su run y métricas, y que el alias `champion` apunta a la versión seleccionada en [03-experimentos](03-experimentos.ipynb).

- Modelo registrado: `workspace.default.pcd_otono_2026_tubecast`
- Registry: `databricks-uc` (en pruebas locales, SQLite con el mismo nombre)
- Registro y alias: `task_registrar_modelos` en `src/entrenamiento/flow.py`

In [1]:
import os, sys, json
from pathlib import Path

# Los notebooks se ejecutan desde la raíz del repositorio para compartir rutas con src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [2]:
from src.entrenamiento.reportes import configurar_mlflow
cfg = configurar_mlflow()
print("Tracking URI :", cfg["tracking_uri"])
print("Registry URI :", cfg["registry_uri"])
print("Experimento  :", cfg["experimento"])
print("Modelo       :", cfg["modelo"], "@" + cfg["alias"])

Tracking URI : sqlite:///mlflow.db
Registry URI : sqlite:///mlflow.db
Experimento  : /Shared/pcd-otono-2026-proyecto-tubecast
Modelo       : workspace.default.pcd_otono_2026_tubecast @champion


## 1. Versiones registradas

In [3]:
from src.entrenamiento.reportes import versiones_registradas, runs_ultima_ejecucion, tabla_comparacion
versiones = versiones_registradas(cfg["modelo"])
versiones

,version,familia,run_id,test_rmse,test_mae,aliases,creada
0,1,ridge,c8fb34793368468c811c914cc9019351,"2,318,929.81","1,763,913.74",,2026-10-27 01:24:15.903
1,2,random-forest,39ee1e5abe07465ca8795074d0706605,"2,168,829.41","1,621,174.87",champion,2026-10-27 01:24:16.160
2,3,gradient-boosting,6b24e4c607524668802f91765b5174b6,"2,257,517.10","1,675,973.32",,2026-10-27 01:24:16.411


Cada versión tiene etiquetas `familia`, `final_run_id`, `parent_run_id`, `val_rmse` y `test_*`, y una descripción con sus métricas; el `run_id` de la versión es el run `<familia>-final` que produjo el artefacto. Así, cada versión se relaciona con su familia, su run y sus métricas.

In [4]:
r = runs_ultima_ejecucion(cfg["experimento"])
comp = tabla_comparacion(r["finales"])
ultimas = versiones[versiones["run_id"].isin(comp["run_id"])]
cruce = ultimas.merge(comp[["run_id", "run", "test_rmse"]], on="run_id", suffixes=("_tag", "_run"))
display(cruce[["version", "familia", "run", "test_rmse_tag", "test_rmse_run", "aliases"]])

assert len(ultimas) >= 3, "Se esperan al menos tres versiones (una por familia) de la última ejecución"
assert np.allclose(cruce["test_rmse_tag"], cruce["test_rmse_run"], rtol=1e-6)
print("Las versiones de la última ejecución coinciden con sus runs -final: OK")

,version,familia,run,test_rmse_tag,test_rmse_run,aliases
0,1,ridge,ridge-final,"2,318,929.81","2,318,929.81",
1,2,random-forest,random-forest-final,"2,168,829.41","2,168,829.41",champion
2,3,gradient-boosting,gradient-boosting-final,"2,257,517.10","2,257,517.10",


Las versiones de la última ejecución coinciden con sus runs -final: OK


## 2. Alias `champion`

In [5]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
champion = client.get_model_version_by_alias(cfg["modelo"], cfg["alias"])
display(Markdown(
    f"`{cfg['modelo']}@{cfg['alias']}` → **versión {champion.version}** "
    f"(familia `{champion.tags.get('familia')}`, run `{champion.run_id}`, "
    f"test_rmse = {float(champion.tags['test_rmse']):,.0f})"
))
mejor = cruce.sort_values("test_rmse_run").iloc[0]
assert str(mejor["version"]) == str(champion.version), "El alias no apunta a la versión con menor test_rmse"
print("El alias champion apunta a la versión con menor test_rmse de la última ejecución: OK")

`workspace.default.pcd_otono_2026_tubecast@champion` → **versión 2** (familia `random-forest`, run `39ee1e5abe07465ca8795074d0706605`, test_rmse = 2,168,829)

El alias champion apunta a la versión con menor test_rmse de la última ejecución: OK


**Criterio de selección.** Menor RMSE en test entre los runs `<familia>-final` de la misma ejecución, bajo la misma partición y presupuesto de HPO. La regla está en código (`task_comparar_modelos`) y se aplica sin fijar la familia ganadora. Si una ejecución futura obtiene un mejor modelo de otra familia, el alias se mueve a esa nueva versión y la API la toma al reiniciar o al llamar `POST /reload`.

## 3. Contenido del artefacto: preprocesamiento, firma y ejemplo de entrada

In [6]:
import mlflow
from mlflow.models import Model

uri = f"models:/{cfg['modelo']}@{cfg['alias']}"
ruta_local = mlflow.artifacts.download_artifacts(uri)
meta = Model.load(ruta_local)
print("Flavors:", list(meta.flavors))
print("Firma de entrada :", meta.signature.inputs)
print("Firma de salida  :", meta.signature.outputs)
ejemplo = meta.load_input_example(ruta_local)
display(ejemplo)

Flavors: ['python_function', 'sklearn']
Firma de entrada : ['lag_14': double (required), 'lag_13': double (required), 'lag_12': double (required), 'lag_11': double (required), 'lag_10': double (required), 'lag_9': double (required), 'lag_8': double (required), 'lag_7': double (required), 'lag_6': double (required), 'lag_5': double (required), 'lag_4': double (required), 'lag_3': double (required), 'lag_2': double (required), 'lag_1': double (required)]
Firma de salida  : [Tensor('float64', (-1,))]


,lag_14,lag_13,lag_12,lag_11,lag_10,lag_9,lag_8,lag_7,lag_6,lag_5,lag_4,lag_3,lag_2,lag_1
0,"3,632,565.00","7,692,959.00","1,759,760.00","2,729,141.00","9,646,087.00","3,448,161.00","3,380,479.00","3,853,999.00","5,150,554.00","1,360,548.00","5,300,909.00","3,785,894.00","1,533,496.00","1,599,046.00"
1,"7,692,959.00","1,759,760.00","2,729,141.00","9,646,087.00","3,448,161.00","3,380,479.00","3,853,999.00","5,150,554.00","1,360,548.00","5,300,909.00","3,785,894.00","1,533,496.00","1,599,046.00","2,849,508.00"
2,"1,759,760.00","2,729,141.00","9,646,087.00","3,448,161.00","3,380,479.00","3,853,999.00","5,150,554.00","1,360,548.00","5,300,909.00","3,785,894.00","1,533,496.00","1,599,046.00","2,849,508.00","2,967,100.00"


In [7]:
pipeline = mlflow.sklearn.load_model(uri)
print(pipeline)
pred_registry = pipeline.predict(ejemplo)
print("Predicción sobre el ejemplo de entrada:", np.round(pred_registry, 0))

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 RandomForestRegressor(max_depth=8, n_estimators=250, n_jobs=-1,
                                       random_state=42))])
Predicción sobre el ejemplo de entrada: [4332824. 3780802. 2134309.]


El artefacto es el `Pipeline` completo (`StandardScaler` + regresor), así que quien lo carga aplica exactamente el mismo preprocesamiento que en entrenamiento. La firma fija el esquema `lag_14 … lag_1` (double) y la salida numérica.

In [8]:
import joblib
ruta_pkl = Path("artifacts/champion_model.pkl")
if ruta_pkl.exists():
    local = joblib.load(ruta_pkl)
    meta_local = json.loads(Path("artifacts/champion_metadata.json").read_text(encoding="utf-8"))
    print("Metadata local:", meta_local)
    assert np.allclose(local.predict(ejemplo), pred_registry)
    print("El respaldo local artifacts/champion_model.pkl produce las mismas predicciones que el Registry: OK")
else:
    print("No hay artefacto local; la API usará únicamente el Registry.")

Metadata local: {'model_name': 'workspace.default.pcd_otono_2026_tubecast', 'alias': 'champion', 'version': '2', 'familia_ganadora': 'random-forest', 'final_run_id': '39ee1e5abe07465ca8795074d0706605', 'tracking_uri': 'sqlite:///mlflow.db', 'test_rmse': 2168829.4104615403, 'test_mae': 1621174.8672469454, 'test_mape': 160.10312414866098}
El respaldo local artifacts/champion_model.pkl produce las mismas predicciones que el Registry: OK


**Procedencia fuera de Git.** `artifacts/champion_model.pkl` y `champion_metadata.json` (ignorados por Git) guardan el mismo pipeline y su versión, run y métricas. Así, quien no tiene credenciales de Databricks puede verificar la API con un artefacto obtenido por el equipo y conocer su origen exacto.

Siguiente: [05-pipeline](05-pipeline.ipynb).